# MWIS Pipeline

This notebook is the clean scheduling core. It loads the Layer E candidate-slot file, builds the MWIS conflict graph, decomposes the graph into QPU-sized pieces, reconstructs the schedule, and saves standardized artifacts for the graphics and QUBO/annealing notebooks.

The original `ground_station_pipeline.ipynb` is intentionally left unchanged.

## 1. Configuration

All run parameters (input file, run name, QPU limits, decomposition budgets, exact-solver limits) are read from `results/mwis/mwis_config.json`. Edit that file to change the experiment. A copy of the configuration used is saved as `run_config.json` in the output folder `results/runs/mwis_pipeline/<run_name>`.

In [ ]:
from __future__ import annotations

import json
import math
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"


# --------- loading configuration environment ----------
# load .json config file
CONFIG_PATH = RESULTS_DIR / "_config.json"
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# load running environment
CANDIDATE_SLOT_CSV = PROJECT_ROOT / run_config["candidate_slot_csv"]

# idk
RUN_NAME = run_config["run_name_mwis"]
RUN_DIR = RESULTS_DIR / "runs" / "mwis_pipeline" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# QPU physical constraints
MAX_QPU_TERMINAL_NODES = run_config["max_qpu_terminal_nodes"]
MAX_QPU_TERMINAL_COUPLERS = run_config["max_qpu_terminal_couplers"]

# Separator-search limits. 
# If these limits are exceeded, the unresolved part is solved heuristically and clearly reported as non-exact.
MAX_SEPARATOR_SIZE = run_config["max_separator_size"]
MAX_SEPARATOR_CANDIDATE_POOL = run_config["max_separator_candidate_pool"]
MAX_SEPARATOR_STATES = run_config["max_separator_states"]
MAX_RECURSION_DEPTH = run_config["max_recursion_depth"]
MAX_DECOMPOSITION_NODES = run_config["max_decomposition_nodes"]

# Exact classical terminal-solver guards.
EXACT_ENUMERATION_NODE_LIMIT = run_config["exact_enumeration_node_limit"]
EXACT_CLIQUE_NODE_LIMIT = run_config["exact_clique_node_limit"]

# loggin environment configuration
(RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"Artifacts will be saved to: {RUN_DIR.resolve()}")

Loaded configuration from: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\mwis\mwis_config.json
Artifacts will be saved to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\runs\mwis_pipeline\latest_mwis_run


## 2. Load candidate slots

The data-wrangling notebooks already created weighted candidate contacts. Here we keep the essential columns used by the MWIS model.

In [3]:
REQUIRED_COLUMNS = [
    "slot_id", "source_interval_id", "satellite_id", "antenna_id", "station_id",
    "start_time_hours", "end_time_hours", "duration_hours",
    "reconfiguration_time_hours", "priority", "urgency", "weight",
]

slots = pd.read_csv(CANDIDATE_SLOT_CSV)[REQUIRED_COLUMNS]
slots.to_csv(RUN_DIR / "candidate_slots.csv", index=False)

print({
    "candidate_slots": len(slots),
    "satellites": slots["satellite_id"].nunique(),
    "antennas": slots["antenna_id"].nunique(),
    "source_intervals": slots["source_interval_id"].nunique(),
})

{'candidate_slots': 533, 'satellites': 81, 'antennas': 6, 'source_intervals': 426}


## 3. Build the MWIS conflict graph

Each candidate contact becomes one node. An edge means two contacts cannot both be selected. Therefore, a feasible schedule is an independent set, and the best schedule is a Maximum Weight Independent Set.

In [ ]:
# ----------------------------------------------------------
# Build conflig graph
# ----------------------------------------------------------

def intervals_overlap(a_start: float, a_end: float, b_start: float, b_end: float) -> bool:
    """
    Check whether two intervals (given the extremes) overlap
    """
    return a_start < b_end and b_start < a_end


def gap_between(a: pd.Series, b: pd.Series) -> float:
    """
    Return the idel gap between two non-ovelapping intervals
    """
    return b["start_time_hours"] - a["end_time_hours"] \
        if a["end_time_hours"] <= b["start_time_hours"]  \
            else a["start_time_hours"] - b["end_time_hours"] 


def add_conflict_edge(graph: nx.Graph, u: str, v: str, reason: str, **attrs: Any) -> None:
    """
    Add conflicting edge in the graph
    """
    if u == v or graph.has_edge(u, v):
        return
    graph.add_edge(u, v, reason=reason, **attrs)


def build_conflict_graph(slot_df: pd.DataFrame) -> nx.Graph:
    graph = nx.Graph()

    # each node represent a condiate contact slot
    for row in slot_df.itertuples(index=False):
        graph.add_node(
            row.slot_id,
            source_interval_id=row.source_interval_id,
            satellite_id=str(row.satellite_id),
            antenna_id=row.antenna_id,
            station_id=row.station_id,
            start_time=float(row.start_time_hours),
            end_time=float(row.end_time_hours),
            duration=float(row.duration_hours),
            reconfiguration_time=float(row.reconfiguration_time_hours),
            priority=float(row.priority),
            urgency=float(row.urgency),
            weight=float(row.weight),
        )

    # set those contant slots deriving from the same visibility window as conflicting
    for source_interval_id, group in slot_df.groupby("source_interval_id", sort=False):
        ids = group["slot_id"].tolist()
        for u, v in combinations(ids, 2):
            add_conflict_edge(
                graph, u, v,
                "same communication request",
                source_interval_id=source_interval_id,
            )

    # Same-antenna temporal conflicts.
    for _, group in slot_df.groupby("antenna_id", sort=False):
        antenna_slots = group.sort_values("start_time_hours").reset_index(drop=True)
        rows = [row for _, row in antenna_slots.iterrows()]
        # for each pair of contant slots with the same antenna ...
        for i, first in enumerate(rows):
            for second in rows[i + 1:]:

                # check whether the two intervals overlap
                if intervals_overlap(
                    first["start_time_hours"], first["end_time_hours"],
                    second["start_time_hours"], second["end_time_hours"],
                ):
                    add_conflict_edge(graph, first["slot_id"], second["slot_id"], "same-antenna overlap")

                # if they don't overlap, check whether they're still two close for antenna reconfiguration
                elif gap_between(first, second) < max(first["reconfiguration_time_hours"], second["reconfiguration_time_hours"]):
                    add_conflict_edge(
                        graph, first["slot_id"], second["slot_id"],
                        "same-antenna reconfiguration time",
                        reconfiguration_time=max(first["reconfiguration_time_hours"], second["reconfiguration_time_hours"]),
                    )
    return graph
#endfun build_conflict_graph

conflict_graph = build_conflict_graph(slots)

node_rows = [{"slot_id": node_id, **data} for node_id, data in conflict_graph.nodes(data=True)]
edge_rows = [
    {"source": u, "target": v, **data}
    for u, v, data in conflict_graph.edges(data=True)
]
# store grahp
pd.DataFrame(node_rows).to_csv(RUN_DIR / "conflict_graph_nodes.csv", index=False)
pd.DataFrame(edge_rows).to_csv(RUN_DIR / "conflict_graph_edges.csv", index=False)

print({
    "nodes": conflict_graph.number_of_nodes(),
    "edges_couplers": conflict_graph.number_of_edges(),
    "components": nx.number_connected_components(conflict_graph),
    "edge_reasons": dict(Counter(data["reason"] for _, _, data in conflict_graph.edges(data=True))),
})

{'nodes': 533, 'edges_couplers': 1994, 'components': 23, 'edge_reasons': {'same communication request': 107, 'same-antenna overlap': 1085, 'same-antenna reconfiguration time': 802}}


## 4. Natural connected components

Disconnected components are independent MWIS subproblems. This split is exact because no conflict edge is cut.

In [ ]:
# ----------------------------------------------------------
# Retrive and display connected components information
# ----------------------------------------------------------

# helper function to print cc summary
def component_summary(component_id: str, graph: nx.Graph) -> dict[str, Any]:
    weights = [data["weight"] for _, data in graph.nodes(data=True)]
    return {
        "component_id": component_id,
        "node_count": graph.number_of_nodes(),
        "edge_count": graph.number_of_edges(),
        "density": nx.density(graph) if graph.number_of_nodes() > 1 else 0.0,
        "total_weight": sum(weights),
        "max_weight": max(weights, default=0.0),
        "satellite_count": len({data["satellite_id"] for _, data in graph.nodes(data=True)}),
        "antenna_count": len({data["antenna_id"] for _, data in graph.nodes(data=True)}),
    }


components = [] # where we store connected components

# iterate through the graph and extract CCs
for index, node_ids in enumerate(
    sorted(nx.connected_components(conflict_graph), key=lambda c: (-len(c), sorted(c))),
    start=1,
):
    # print cc summary
    component_id = f"component_{index:03d}"
    graph = conflict_graph.subgraph(sorted(node_ids)).copy()
    components.append({"component_id": component_id, "graph": graph, "summary": component_summary(component_id, graph)})

components_summary = pd.DataFrame([component["summary"] for component in components])
components_summary.to_csv(RUN_DIR / "components_summary.csv", index=False)
components_summary.head(10)

,component_id,node_count,edge_count,density,total_weight,max_weight,satellite_count,antenna_count
0,component_001,85,342,0.095798,5074.014,91.271,52,1
1,component_002,66,282,0.131469,3834.006,82.860,45,1
2,component_003,45,176,0.177778,2795.178,86.232,34,1
3,component_004,40,145,0.185897,2513.540,80.812,33,1
4,component_005,36,157,0.249206,1917.399,82.955,23,1
5,component_006,35,120,0.201681,2436.331,92.283,29,1
6,component_007,32,104,0.209677,2230.881,89.724,28,1
7,component_008,31,153,0.329032,1636.794,79.284,26,1
8,component_009,22,84,0.363636,1331.672,87.167,19,1
9,component_010,17,44,0.323529,1194.713,92.319,13,1


## 5. Exact separator decomposition

Components that already fit the QPU limits become terminal subgraphs. Oversized components are recursively decomposed using small vertex separators. If exact decomposition cannot finish within the configured limits, the unresolved node is marked incomplete.

In [ ]:
def graph_fits_qpu_limits(graph: nx.Graph) -> bool:
    """
    Check whether the given graphs breaks QPU limits or not
    """
    return (
        graph.number_of_nodes() <= MAX_QPU_TERMINAL_NODES
        and graph.number_of_edges() <= MAX_QPU_TERMINAL_COUPLERS
    )


def exact_preprocess(graph: nx.Graph) -> dict[str, Any]:
    """
    Select isolated nodes and solve them exactly
    """
    forced_selected = set(nx.isolates(graph))
    reduced = graph.copy()
    reduced.remove_nodes_from(forced_selected)
    return {
        "graph": reduced,
        "forced_selected": forced_selected,
        "forced_objective": sum(graph.nodes[node]["weight"] for node in forced_selected),
    }


def residual_components_after_separator(graph: nx.Graph, separator: tuple[str, ...]) -> list[set[str]]:
    """
    Get the connected components ofthe graph after removing the 'separator' elements 
    """
    residual = graph.copy() # works on a copy of the network
    residual.remove_nodes_from(separator)
    return [set(c) for c in nx.connected_components(residual)]


def feasible_separator_states(graph: nx.Graph, separator: tuple[str, ...]) -> list[frozenset[str]]:
    states = []
    for size in range(len(separator) + 1):
        for selected in combinations(separator, size):
            selected = set(selected)
            if all(not graph.has_edge(a, b) for a, b in combinations(selected, 2)):
                states.append(frozenset(selected))
    return states


def score_separator(graph: nx.Graph, separator: tuple[str, ...]) -> tuple | None:
    pieces = residual_components_after_separator(graph, separator)
    if len(pieces) <= 1:
        return None
    sizes = [len(piece) for piece in pieces]
    return (
        max(sizes),
        len(feasible_separator_states(graph, separator)),
        max(sizes) - min(sizes),
        len(separator),
        -len(pieces),
        tuple(sorted(separator)),
    )


def find_separator(graph: nx.Graph) -> dict[str, Any] | None:
    candidates = []

    for node in nx.articulation_points(graph):
        separator = (node,)
        score = score_separator(graph, separator)
        if score is not None:
            candidates.append((score, separator, "articulation point"))

    node_ids = sorted(graph.nodes)
    if MAX_SEPARATOR_SIZE >= 2 and len(node_ids) <= 200:
        for separator in combinations(node_ids, 2):
            score = score_separator(graph, separator)
            if score is not None:
                candidates.append((score, separator, "exhaustive separator pair"))

    pool = [
        node for node, _ in sorted(
            graph.degree,
            key=lambda item: (-item[1], item[0]),
        )[:MAX_SEPARATOR_CANDIDATE_POOL]
    ]
    for size in range(2, MAX_SEPARATOR_SIZE + 1):
        for separator in combinations(pool, size):
            score = score_separator(graph, separator)
            if score is not None:
                candidates.append((score, separator, f"degree-pool separator size {size}"))

    if not candidates:
        return None
    score, separator, method = min(candidates)
    return {"separator": tuple(separator), "score": score, "method": method}


def build_decomposition_node(graph: nx.Graph, component_id: str, context: dict[str, Any], depth: int = 0) -> dict[str, Any]:
    cache_key = frozenset(graph.nodes)
    if cache_key in context["cache"]:
        context["cache_hits"] += 1
        return context["cache"][cache_key]

    node_number = context["next_node_number"]
    context["next_node_number"] += 1
    node_id = f"{component_id}_dnode_{node_number:04d}"

    preprocessing = exact_preprocess(graph)
    reduced = preprocessing["graph"]
    node = {
        "decomposition_node_id": node_id,
        "original_component_id": component_id,
        "depth": depth,
        "input_node_count": graph.number_of_nodes(),
        "input_edge_count": graph.number_of_edges(),
        "graph": reduced,
        "forced_selected": set(preprocessing["forced_selected"]),
        "forced_objective": preprocessing["forced_objective"],
        "children": [],
        "branches": [],
    }
    context["cache"][cache_key] = node
    context["nodes"][node_id] = node

    if context["next_node_number"] > MAX_DECOMPOSITION_NODES:
        node.update({"node_type": "incomplete", "reason": "decomposition node budget exceeded"})
        return node
    if depth > MAX_RECURSION_DEPTH:
        node.update({"node_type": "incomplete", "reason": "recursion depth exceeded"})
        return node
    if reduced.number_of_nodes() == 0:
        node.update({"node_type": "terminal", "terminal_reason": "resolved by exact preprocessing"})
        return node

    natural_pieces = [set(c) for c in nx.connected_components(reduced)]
    if len(natural_pieces) > 1:
        node.update({"node_type": "independent_sum"})
        node["children"] = [
            build_decomposition_node(reduced.subgraph(sorted(piece)).copy(), component_id, context, depth + 1)
            for piece in sorted(natural_pieces, key=lambda c: (-len(c), sorted(c)))
        ]
        return node

    if graph_fits_qpu_limits(reduced):
        node.update({"node_type": "terminal", "terminal_reason": "fits QPU limits"})
        return node

    separator_result = find_separator(reduced)
    if separator_result is None:
        node.update({"node_type": "incomplete", "reason": "no separator found within configured search limits"})
        return node

    separator = tuple(separator_result["separator"])
    states = feasible_separator_states(reduced, separator)
    if context["separator_state_count"] + len(states) > MAX_SEPARATOR_STATES:
        node.update({"node_type": "incomplete", "reason": "separator-state budget exceeded"})
        return node

    context["separator_state_count"] += len(states)
    node.update({
        "node_type": "separator",
        "separator": separator,
        "separator_method": separator_result["method"],
        "separator_score": separator_result["score"],
    })

    separator_set = set(separator)
    for state_index, selected_separator in enumerate(states):
        selected_separator = set(selected_separator)
        forced_neighbors = set()
        for selected_node in selected_separator:
            forced_neighbors.update(reduced.neighbors(selected_node))
        forced_neighbors.difference_update(separator_set)

        branch_graph = reduced.copy()
        branch_graph.remove_nodes_from(separator_set | forced_neighbors)
        branch_components = [set(c) for c in nx.connected_components(branch_graph)]
        children = [
            build_decomposition_node(branch_graph.subgraph(sorted(piece)).copy(), component_id, context, depth + 1)
            for piece in sorted(branch_components, key=lambda c: (-len(c), sorted(c)))
        ]
        node["branches"].append({
            "state_index": state_index,
            "selected_separator": selected_separator,
            "state_objective": sum(reduced.nodes[n]["weight"] for n in selected_separator),
            "children": children,
        })
    return node


def build_component_decomposition(component: dict[str, Any]) -> dict[str, Any]:
    context = {"next_node_number": 1, "cache": {}, "nodes": {}, "cache_hits": 0, "separator_state_count": 0}
    root = build_decomposition_node(component["graph"], component["component_id"], context)
    nodes = list(context["nodes"].values())
    terminals = [node for node in nodes if node["node_type"] == "terminal"]
    incomplete = [node for node in nodes if node["node_type"] == "incomplete"]
    return {
        "component_id": component["component_id"],
        "root": root,
        "nodes": nodes,
        "terminals": terminals,
        "incomplete": incomplete,
        "status": "complete" if not incomplete else "incomplete",
        "separator_state_count": context["separator_state_count"],
        "cache_hits": context["cache_hits"],
        "max_depth": max((node["depth"] for node in nodes), default=0),
    }

In [6]:
decompositions = [build_component_decomposition(component) for component in components]
terminal_nodes = [node for decomp in decompositions for node in decomp["terminals"]]
nonempty_terminal_nodes = [node for node in terminal_nodes if node["graph"].number_of_nodes() > 0]
incomplete_nodes = [node for decomp in decompositions for node in decomp["incomplete"]]

decomposition_rows = []
for decomp in decompositions:
    root = decomp["root"]
    decomposition_rows.append({
        "component_id": decomp["component_id"],
        "status": decomp["status"],
        "root_type": root["node_type"],
        "input_nodes": root["input_node_count"],
        "input_edges": root["input_edge_count"],
        "terminal_graphs": len([n for n in decomp["terminals"] if n["graph"].number_of_nodes() > 0]),
        "incomplete_nodes": len(decomp["incomplete"]),
        "separator_states": decomp["separator_state_count"],
        "cache_hits": decomp["cache_hits"],
        "max_depth": decomp["max_depth"],
    })

pd.DataFrame(decomposition_rows).to_csv(RUN_DIR / "decomposition_summary.csv", index=False)

def serialize_graph(graph: nx.Graph) -> dict[str, Any]:
    return {
        "nodes": [{"id": node, **data} for node, data in graph.nodes(data=True)],
        "edges": [{"source": u, "target": v, **data} for u, v, data in graph.edges(data=True)],
    }

terminal_subgraphs = []
for node in nonempty_terminal_nodes:
    terminal_subgraphs.append({
        "terminal_id": node["decomposition_node_id"],
        "original_component_id": node["original_component_id"],
        "depth": node["depth"],
        "node_count": node["graph"].number_of_nodes(),
        "edge_count": node["graph"].number_of_edges(),
        "forced_objective": node["forced_objective"],
        "terminal_reason": node["terminal_reason"],
        "graph": serialize_graph(node["graph"]),
    })

(RUN_DIR / "terminal_subgraphs.json").write_text(json.dumps(terminal_subgraphs, indent=2), encoding="utf-8")

print({
    "natural_components": len(components),
    "complete_decompositions": sum(d["status"] == "complete" for d in decompositions),
    "incomplete_decompositions": sum(d["status"] != "complete" for d in decompositions),
    "terminal_subgraphs": len(nonempty_terminal_nodes),
    "largest_terminal_nodes": max((n["graph"].number_of_nodes() for n in nonempty_terminal_nodes), default=0),
    "largest_terminal_couplers": max((n["graph"].number_of_edges() for n in nonempty_terminal_nodes), default=0),
    "separator_states": sum(d["separator_state_count"] for d in decompositions),
})

{'natural_components': 23, 'complete_decompositions': 23, 'incomplete_decompositions': 0, 'terminal_subgraphs': 72, 'largest_terminal_nodes': 29, 'largest_terminal_couplers': 140, 'separator_states': 41}


## 6. MWIS solvers and reconstruction

Terminal subgraphs are solved exactly when possible. If decomposition is incomplete, only the unresolved part is solved with a deterministic weighted-greedy heuristic. The heuristic result is feasible, but it is reported as a lower bound with a conservative upper bound.

In [7]:
def node_set_weight(graph: nx.Graph, selected: set[str]) -> float:
    return sum(graph.nodes[node]["weight"] for node in selected)


def is_independent(graph: nx.Graph, selected: set[str]) -> bool:
    return all(not graph.has_edge(u, v) for u, v in combinations(selected, 2))


def solve_mwis_enumeration(graph: nx.Graph) -> dict[str, Any]:
    nodes = sorted(graph.nodes)
    best_nodes = set()
    best_value = 0.0
    for size in range(len(nodes) + 1):
        for subset in combinations(nodes, size):
            selected = set(subset)
            if is_independent(graph, selected):
                value = node_set_weight(graph, selected)
                if value > best_value:
                    best_nodes, best_value = selected, value
    return {"status": "optimal", "selected_nodes": best_nodes, "objective": best_value, "solver": "exact enumeration"}


def solve_mwis_clique(graph: nx.Graph) -> dict[str, Any]:
    complement = nx.complement(graph)
    for node, data in graph.nodes(data=True):
        complement.nodes[node]["scaled_weight"] = int(round(data["weight"] * 1000))
    selected, _ = nx.algorithms.clique.max_weight_clique(complement, weight="scaled_weight")
    selected = set(selected)
    return {
        "status": "optimal",
        "selected_nodes": selected,
        "objective": node_set_weight(graph, selected),
        "solver": "complement max-weight clique",
    }


def solve_mwis_greedy(graph: nx.Graph) -> dict[str, Any]:
    remaining = set(graph.nodes)
    selected = set()
    while remaining:
        best = max(
            remaining,
            key=lambda node: (
                graph.nodes[node]["weight"] / (1 + graph.degree(node)),
                graph.nodes[node]["weight"],
                -graph.degree(node),
                node,
            ),
        )
        selected.add(best)
        remaining.remove(best)
        remaining.difference_update(graph.neighbors(best))
    return {
        "status": "heuristic_feasible",
        "selected_nodes": selected,
        "objective": node_set_weight(graph, selected),
        "solver": "deterministic weighted greedy",
    }


def solve_terminal_mwis(graph: nx.Graph) -> dict[str, Any]:
    if graph.number_of_nodes() == 0:
        return {"status": "optimal", "selected_nodes": set(), "objective": 0.0, "solver": "empty graph"}
    if graph.number_of_nodes() <= EXACT_ENUMERATION_NODE_LIMIT:
        return solve_mwis_enumeration(graph)
    if graph.number_of_nodes() <= EXACT_CLIQUE_NODE_LIMIT:
        return solve_mwis_clique(graph)
    return solve_mwis_greedy(graph)


def conservative_upper_bound(graph: nx.Graph) -> float:
    return sum(data["weight"] for _, data in graph.nodes(data=True))

In [8]:
def evaluate_node(node: dict[str, Any], memo: dict[str, dict[str, Any]] | None = None) -> dict[str, Any]:
    if memo is None:
        memo = {}
    node_id = node["decomposition_node_id"]
    if node_id in memo:
        return memo[node_id]

    base_selected = set(node.get("forced_selected", set()))
    base_objective = float(node.get("forced_objective", 0.0))

    if node["node_type"] == "incomplete":
        result = solve_mwis_greedy(node["graph"])
        output = {
            "status": "heuristic_feasible",
            "selected_nodes": base_selected | result["selected_nodes"],
            "objective": base_objective + result["objective"],
            "upper_bound": base_objective + conservative_upper_bound(node["graph"]),
            "exact_objective": 0.0,
            "heuristic_objective": result["objective"],
            "terminal_ids": set(),
            "heuristic_ids": {node_id},
            "fallback_details": [{
                "node_id": node_id,
                "component_id": node["original_component_id"],
                "reason": node["reason"],
                "node_count": node["graph"].number_of_nodes(),
                "edge_count": node["graph"].number_of_edges(),
                "heuristic_objective": result["objective"],
                "upper_bound": base_objective + conservative_upper_bound(node["graph"]),
            }],
        }
        memo[node_id] = output
        return output

    if node["node_type"] == "terminal":
        result = solve_terminal_mwis(node["graph"])
        is_exact = result["status"] == "optimal"
        objective = base_objective + result["objective"]
        output = {
            "status": "optimal" if is_exact else "heuristic_feasible",
            "selected_nodes": base_selected | result["selected_nodes"],
            "objective": objective,
            "upper_bound": objective if is_exact else base_objective + conservative_upper_bound(node["graph"]),
            "exact_objective": result["objective"] if is_exact else 0.0,
            "heuristic_objective": 0.0 if is_exact else result["objective"],
            "terminal_ids": {node_id} if node["graph"].number_of_nodes() and is_exact else set(),
            "heuristic_ids": set(),
            "fallback_details": [],
        }
        memo[node_id] = output
        return output

    if node["node_type"] == "independent_sum":
        children = [evaluate_node(child, memo) for child in node["children"]]
        output = {
            "status": "optimal" if all(child["status"] == "optimal" for child in children) else "heuristic_feasible",
            "selected_nodes": base_selected | set().union(*(child["selected_nodes"] for child in children)),
            "objective": base_objective + sum(child["objective"] for child in children),
            "upper_bound": base_objective + sum(child["upper_bound"] for child in children),
            "exact_objective": sum(child["exact_objective"] for child in children),
            "heuristic_objective": sum(child["heuristic_objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)),
            "fallback_details": [detail for child in children for detail in child["fallback_details"]],
        }
        memo[node_id] = output
        return output

    branch_results = []
    branch_upper_bounds = []
    for branch in node["branches"]:
        children = [evaluate_node(child, memo) for child in branch["children"]]
        selected = base_selected | set(branch["selected_separator"])
        for child in children:
            selected.update(child["selected_nodes"])

        objective = base_objective + branch["state_objective"] + sum(child["objective"] for child in children)
        upper_bound = base_objective + branch["state_objective"] + sum(child["upper_bound"] for child in children)
        branch_upper_bounds.append(upper_bound)
        branch_results.append({
            "status": "optimal" if all(child["status"] == "optimal" for child in children) else "heuristic_feasible",
            "selected_nodes": selected,
            "objective": objective,
            "upper_bound": upper_bound,
            "exact_objective": branch["state_objective"] + sum(child["exact_objective"] for child in children),
            "heuristic_objective": sum(child["heuristic_objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)) if children else set(),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)) if children else set(),
            "fallback_details": [detail for child in children for detail in child["fallback_details"]],
        })

    best = max(branch_results, key=lambda item: (item["objective"], sorted(item["selected_nodes"])))
    best["upper_bound"] = max(branch_upper_bounds, default=best["objective"])
    if best["upper_bound"] > best["objective"] + 1e-9:
        best["status"] = "heuristic_feasible"
    memo[node_id] = best
    return best


component_results = [evaluate_node(decomp["root"]) for decomp in decompositions]
selected_nodes = set().union(*(result["selected_nodes"] for result in component_results))
final_objective = sum(result["objective"] for result in component_results)
final_upper_bound = sum(result["upper_bound"] for result in component_results)

solution = {
    "status": "optimal" if all(result["status"] == "optimal" for result in component_results) else "heuristic_feasible",
    "selected_nodes": selected_nodes,
    "objective": final_objective,
    "upper_bound": final_upper_bound,
    "absolute_gap_bound": final_upper_bound - final_objective,
    "relative_gap_bound_percent": 0.0 if final_upper_bound == 0 else 100 * (final_upper_bound - final_objective) / final_upper_bound,
    "exact_component_count": sum(result["status"] == "optimal" for result in component_results),
    "heuristic_component_count": sum(result["status"] != "optimal" for result in component_results),
    "exact_components_objective_sum": sum(result["objective"] for result in component_results if result["status"] == "optimal"),
    "heuristic_components_objective_sum": sum(result["objective"] for result in component_results if result["status"] != "optimal"),
    "heuristic_components_upper_bound": sum(result["upper_bound"] for result in component_results if result["status"] != "optimal"),
    "terminal_ids": set().union(*(result["terminal_ids"] for result in component_results)),
    "heuristic_ids": set().union(*(result["heuristic_ids"] for result in component_results)),
    "fallback_details": [detail for result in component_results for detail in result["fallback_details"]],
}

if not is_independent(conflict_graph, solution["selected_nodes"]):
    raise AssertionError("Reconstructed solution is not an independent set.")

selected_slots = slots[slots["slot_id"].isin(solution["selected_nodes"])].copy()
selected_slots.to_csv(RUN_DIR / "mwis_solution.csv", index=False)
pd.DataFrame(solution["fallback_details"]).to_csv(RUN_DIR / "heuristic_fallback_details.csv", index=False)

print({
    "status": solution["status"],
    "selected_contacts": len(solution["selected_nodes"]),
    "objective": solution["objective"],
    "upper_bound": solution["upper_bound"],
    "relative_gap_bound_percent": solution["relative_gap_bound_percent"],
    "heuristic_subgraphs": len(solution["heuristic_ids"]),
})

{'status': 'optimal', 'selected_contacts': 111, 'objective': 7882.069, 'upper_bound': 7882.069, 'relative_gap_bound_percent': 0.0, 'heuristic_subgraphs': 0}


## 7. Final summary table

This table is the main compact output of the notebook. The graphics and QUBO notebooks read the artifacts saved above instead of rebuilding this pipeline.

In [ ]:
direct_component_ids = {
    decomp["component_id"]
    for decomp in decompositions
    if decomp["root"]["node_type"] == "terminal"
}
directly_solved_terminals = [
    node for node in nonempty_terminal_nodes
    if node["original_component_id"] in direct_component_ids
]
separator_terminals = [
    node for node in nonempty_terminal_nodes
    if node["original_component_id"] not in direct_component_ids
]

final_summary_rows = [
    ("Number of components (total)", len(components), "Natural connected components."),
    ("Number of couplers (total)", conflict_graph.number_of_edges(), "Conflict edges in the full graph."),
    ("Total number of solution subgraphs", len(directly_solved_terminals) + len(separator_terminals) + len(solution["heuristic_ids"]), "Direct terminals + separator terminals + heuristic fallback subgraphs."),
    ("Subgraphs solved directly", len(directly_solved_terminals), "Natural components that already fit the QPU limits."),
    ("Subgraphs solved exactly with separator decomposition", len(separator_terminals), "Terminal subgraphs produced from oversized components."),
    ("Subgraphs solved heuristically", len(solution["heuristic_ids"]), "Incomplete decomposition nodes solved by fallback heuristic."),
    ("Exact components objective sum", solution["exact_components_objective_sum"], "Objective contribution from exact natural components."),
    ("Exact global objective", solution["objective"] if solution["status"] == "optimal" else "not certified", "Available only when no heuristic fallback was used."),
    ("Incomplete/heuristic components feasible sum", solution["heuristic_components_objective_sum"], "Feasible lower-bound contribution from heuristic components."),
    ("Incomplete/heuristic components conservative upper bound", solution["heuristic_components_upper_bound"], "Conservative upper bound for heuristic components."),
    ("Total feasible objective / lower bound", solution["objective"], "Objective of the reconstructed feasible schedule."),
    ("Total conservative upper bound", solution["upper_bound"], "Conservative upper bound on the true optimum."),
    ("Absolute optimality-gap bound", solution["absolute_gap_bound"], "Upper bound minus feasible objective."),
    ("Relative optimality-gap bound (%)", solution["relative_gap_bound_percent"], "Gap as a percentage of the conservative upper bound."),
]

final_summary = pd.DataFrame(final_summary_rows, columns=["metric", "value", "meaning"])
final_summary.to_csv(RUN_DIR / "final_summary.csv", index=False)
final_summary

,metric,value,meaning
0,Number of components (total),23.000,Natural connected components.
1,Number of couplers (total),1994.000,Conflict edges in the full graph.
2,Total number of solution subgraphs,72.000,Direct terminals + separator terminals + heuri...
3,Subgraphs solved directly,15.000,Natural components that already fit the QPU li...
4,Subgraphs solved exactly with separator decomp...,57.000,Terminal subgraphs produced from oversized com...
5,Subgraphs solved heuristically,0.000,Incomplete decomposition nodes solved by fallb...
6,Exact components objective sum,7882.069,Objective contribution from exact natural comp...
7,Exact global objective,7882.069,Available only when no heuristic fallback was ...
8,Incomplete/heuristic components feasible sum,0.000,Feasible lower-bound contribution from heurist...
9,Incomplete/heuristic components conservative u...,0.000,Conservative upper bound for heuristic compone...
